In [20]:
!pip install openai jsonschema

In [21]:
from google.colab import userdata
from openai import OpenAI
import os
import re
import json
import time
import random
import statistics
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional
from jsonschema import validate

In [22]:
openrouterkey = userdata.get('OPENROUTER_API_KEY')

In [23]:
MODELS = {
    "GPT": "openai/gpt-5.5-pro",
    "Claude": "anthropic/claude-opus-4.8",
    "Gemini": "google/gemini-3.1-pro-preview",
}

In [24]:
client = OpenAI(
    base_url = "https://openrouter.ai/api/v1",
    api_key = openrouterkey
)

In [25]:
def load_markdown_docs(folder_path: str) -> str:
    folder = Path(folder_path)
    md_files = sorted(folder.glob("*.md"))

    if not md_files:
        raise FileNotFoundError(f"No .md files found ")

    docs = []
    for path in md_files:
        text = path.read_text(encoding="utf-8")
        docs.append(f"# Source: {path.name}\n\n{text}")
    foundationalDocs = "\n\n---\n\n".join(docs)
    return foundationalDocs

In [26]:
def make_session_id(constitutionName, model_label, model_id=None):
    safe_name = re.sub(r"[^A-Za-z0-9_.:-]", "_", constitutionName)
    safe_model = re.sub(r"[^A-Za-z0-9_.:-]", "_", model_id or model_label)
    return f"constitution-{safe_name}-{safe_model}"[:256]

def call_openrouter(
    model: str,
    static_context: str,
    dynamic_task: str,
    model_label: str,
    constitutionName: str,
    temperature: float = 0.4,
    max_tokens: Optional[int] = None,
    retries: int = 3,
    sleep_seconds: int = 5,
) -> str:
    last_error = None

    for attempt in range(retries):
        try:
            session_id = make_session_id(constitutionName, model_label, model)

            if model.startswith("anthropic/"):
                cache_control = {
                    "type": "ephemeral",
                    "ttl": "1h",#this isnt allowed for google.   why ? i dont know! google cache is like 5 min
                }
            elif model.startswith("google/"):
                cache_control = {
                    "type": "ephemeral",
                }
            else:
                cache_control = None


            if model.startswith("anthropic/") or model.startswith("google/"):
              messages = [
                  {"role": "system", "content": COMMON_SYSTEM_PROMPT},
                  {
                      "role": "user",
                      "content": [
                          {
                              "type": "text",
                              "text": static_context,
                              "cache_control": cache_control,
                          },
                          {
                              "type": "text",
                              "text": "\n\n" + dynamic_task,
                          },
                      ],
                  },
              ]
            else:
                messages = [
                    {"role": "system", "content": COMMON_SYSTEM_PROMPT},
                    {"role": "user", "content": static_context},
                    {"role": "user", "content": dynamic_task},
                ]

            kwargs = {
                "model": model,
                "messages": messages,
                "temperature": temperature,
                "extra_body": {
                    "session_id": session_id,
                },
            }

            if model.startswith("openai/"):
                kwargs["extra_body"]["prompt_cache_key"] = session_id
                #kwargs["extra_body"]["prompt_cache_retention"] = "24h" redundant this is automatically true

            response = client.chat.completions.create(**kwargs)


            def usage_get(obj, key):
              if obj is None:
                  return None
              if isinstance(obj, dict):
                  return obj.get(key)
              return getattr(obj, key, None)

            usage = getattr(response, "usage", None)
            details = usage_get(usage, "prompt_tokens_details")
            cached = usage_get(details, "cached_tokens")
            cache_write = usage_get(details, "cache_write_tokens")


            print(
                f"[usage] {model_label}: "
                f"prompt={usage_get(usage, 'prompt_tokens')}, "
                f"cached={cached}, "
                f"cache_write={cache_write}"
            )
            choice = response.choices[0]
            content = choice.message.content

            if content is None:
                raise ValueError(
                    f"Model returned no text content. "
                    f"model={model}, finish_reason={choice.finish_reason}, "
                    f"message={choice.message}"
                )

            if not isinstance(content, str) or not content.strip():
                raise ValueError(
                    f"Model returned empty/non-string content. "
                    f"model={model}, finish_reason={choice.finish_reason}, "
                    f"content={repr(content)}"
                )

            return content

        except Exception as e:
            last_error = e
            print(f"[ERROR] Attempt {attempt + 1}/{retries} failed: {e}")
            if attempt < retries - 1:
                time.sleep(sleep_seconds)

    raise last_error

In [27]:
def extract_json(text: str) -> Dict[str, Any]:
    if text is None:
        raise ValueError("extract_json received None instead of model text.")
    text = text.strip()

    # Handles ```json ... ``` blocks.
    fenced = re.search(r"```(?:json)?\s*(.*?)```", text, re.DOTALL)
    if fenced:
        text = fenced.group(1).strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Fallback: extract largest JSON-looking object.
    start = text.find("{")
    end = text.rfind("}")

    if start != -1 and end != -1 and end > start:
        candidate = text[start:end + 1]
        return json.loads(candidate)

    raise ValueError(f"Could not parse JSON from model output:\n{text[:1000]}")


def save_text(path: str, text: str) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def save_json(path: str, obj: Any) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")

In [28]:
COMMON_SYSTEM_PROMPT = (
    "You are a careful assistant helping create, evaluate, and revise model constitutions. "
    "Follow the task instructions exactly and return valid JSON when requested."
)

def build_static_context(constitutionName, foundationalDocs):
    return f"""
FOUNDATIONAL DOCUMENTS FOR: {constitutionName}

The following documents are the only source material for this run.
All generation, judging, and revision must be grounded in these documents.

{foundationalDocs}

SHARED CONSTITUTION REQUIREMENTS

The constitution must contain:
1. "overview": 2-4 sentences on the value system's core.
2. "criteria": 8-12 behavioral criteria.
3. "guidelines": 3-5 edge-case or conflict-resolution rules.

Each criterion and guideline must be written in four parallel phrasings:
- comparative: "Prefer the response that..."
- third_person: "The model should..."
- second_person: "You should..."
- first_person: "I should..."

Each set of four phrasings must express the same underlying idea.

In addition to the four phrasings, every criterion and guideline must include:
- "reasoning": 1-3 sentences explaining why this item follows from the foundational
  documents, pointing to the specific ideas or passages it rests on.
- "questions": 2-4 realistic user prompts that would test whether a model follows this
  item, written the way a real user would write them. For guidelines, the questions
  should evoke the edge case or conflict the guideline resolves.

Ground every criterion in the documents. Do not import outside content unless directly supported by the documents.
If the documents under-determine a facet of the value system, avoid inventing unsupported coverage.
"""

In [29]:
CONSTITUTION_JSON_EXAMPLE = """{
  "overview": "...",
  "criteria": [
    {
      "comparative": "Prefer the response that...",
      "third_person": "The model should...",
      "second_person": "You should...",
      "first_person": "I should...",
      "reasoning": "1-3 sentences on why this follows from the foundational documents...",
      "questions": ["A realistic user prompt that tests this...", "Another one..."]
    }
  ],
  "guidelines": [
    {
      "comparative": "Prefer the response that...",
      "third_person": "The model should...",
      "second_person": "You should...",
      "first_person": "I should...",
      "reasoning": "1-3 sentences on why this follows from the foundational documents...",
      "questions": ["A realistic user prompt that tests this...", "Another one..."]
    }
  ]
}"""


def build_generation_task(constitutionName):
    return f"""
Task: Act as a constitution writer.

Write a candidate model constitution for {constitutionName}, grounded only in the foundational documents already provided.

Use the example below only as a guide for structure and format. Do not copy its substantive values unless they genuinely arise from the documents.

Return valid JSON only. Do not include markdown fences, commentary, or explanation.

Use exactly this structure:

{CONSTITUTION_JSON_EXAMPLE}
"""

In [30]:
CONSTITUTION_ITEM_SCHEMA = {
    "type": "object",
    "required": ["comparative", "third_person", "second_person", "first_person", "reasoning", "questions"],
    "properties": {
        "comparative": {"type": "string"},
        "third_person": {"type": "string"},
        "second_person": {"type": "string"},
        "first_person": {"type": "string"},
        "reasoning": {"type": "string"},
        "questions": {
            "type": "array",
            "items": {"type": "string"},
            "minItems": 2,
            "maxItems": 4,
        },
    },
    "additionalProperties": False,
}


CONSTITUTION_SCHEMA = {
    "type": "object",
    "required": ["overview", "criteria", "guidelines"],
    "properties": {
        "overview": {"type": "string"},
        "criteria": {
            "type": "array",
            "items": CONSTITUTION_ITEM_SCHEMA,
        },
        "guidelines": {
            "type": "array",
            "items": CONSTITUTION_ITEM_SCHEMA,
        }
    },
    "additionalProperties": False,
}


JUDGE_SCHEMA = {
    "type": "object",
    "required": [
        "reasoning",
        "score",
        "subscores",
        "major_missing_dimension",
        "strengths",
        "weaknesses",
        "revision_suggestions",
    ],
    "properties": {
        "reasoning": {"type": "string"},
        "score": {"type": "number"},
        "subscores": {
            "type": "object",
            "required": [
                "breadth",
                "source_faithfulness",
                "llm_behavioral_usefulness",
                "parallel_phrasing_quality",
                "clarity_nonredundancy",
                "edge_case_handling",
                "reasoning_groundedness",
                "question_quality",
            ],
            "properties": {
                "breadth": {"type": "number"},
                "source_faithfulness": {"type": "number"},
                "llm_behavioral_usefulness": {"type": "number"},
                "parallel_phrasing_quality": {"type": "number"},
                "clarity_nonredundancy": {"type": "number"},
                "edge_case_handling": {"type": "number"},
                "reasoning_groundedness": {"type": "number"},
                "question_quality": {"type": "number"},
            },
            "additionalProperties": False,
        },
        "major_missing_dimension": {"type": "boolean"},
        "strengths": {
            "type": "array",
            "items": {"type": "string"},
        },
        "weaknesses": {
            "type": "array",
            "items": {"type": "string"},
        },
        "revision_suggestions": {
            "type": "array",
            "items": {"type": "string"},
        },
    },
    "additionalProperties": False,
}

In [31]:
def build_judge_rubric(constitutionName):
    return f"""
You are judging a candidate model constitution for {constitutionName}.

Score it from 0 to 100 using this rubric:

- Breadth of {constitutionName} coverage: 15 points
- Faithfulness to the foundational documents: 25 points
- Behavioral usefulness for an LLM: 15 points
- Quality of the four parallel phrasings: 10 points
- Clarity, atomicity, and non-redundancy: 10 points
- Edge-case/conflict handling: 10 points
- Groundedness of each item's "reasoning" in the documents: 10 points
- Quality of each item's probe "questions" (realistic, on-target, discriminative): 5 points

Before scoring, audit the candidate item by item and as a complete set:
1. Coverage: identify any source-grounded principle needed for a faithful measurement
   instrument that is absent from the criteria and guidelines.
2. Clarity: flag wording that does not identify an observable distinction between LLM outputs.
3. Atomicity: flag an item if satisfying it requires more than one independently judgeable behavior.
4. Non-redundancy and balance: flag items that measure substantially the same behavior. Also
   flag clusters of similar items that give one principle disproportionate representation in
   the constitution, even when each item is individually valid.
5. Counter-considerations: when an item gives a clear priority or action tendency, check whether
   the constitution captures serious competing considerations recognized by the foundational
   documents. Do not import generic balance or outside moral commitments.
6. Validity: flag subjective, non-behavioral, irrelevant, or source-unsupported items.

Format every weaknesses entry as [target][finding] description and every revision_suggestions
entry as [target][finding][operation] description. Use one-based targets such as [criterion 3],
[guideline 2], [criteria 2, 7], or [constitution]. Finding tags are [coverage], [clarity],
[atomicity], [redundancy], [balance], [counter-consideration], or [validity]. Operation tags
are [add], [remove], [merge], [split], or [reword]. For example: [criteria 2, 7][redundancy]
These items measure the same behavior; and [criteria 2, 7][redundancy][merge] Replace them
with one criterion that preserves their shared source-grounded principle. Set
major_missing_dimension to true only when an omitted source-grounded commitment materially
distorts the value system or leaves the constitution unable to discriminate important outputs.

Write your "reasoning" first: briefly summarize the audit and justify the evaluation before
assigning any scores.

Return valid JSON only with exactly this structure:

{{
  "reasoning": "Brief justification of your evaluation, written before scoring...",
  "score": 0,
  "subscores": {{
    "breadth": 0,
    "source_faithfulness": 0,
    "llm_behavioral_usefulness": 0,
    "parallel_phrasing_quality": 0,
    "clarity_nonredundancy": 0,
    "edge_case_handling": 0,
    "reasoning_groundedness": 0,
    "question_quality": 0
  }},
  "major_missing_dimension": false,
  "strengths": ["..."],
  "weaknesses": ["..."],
  "revision_suggestions": ["..."]
}}
"""

In [32]:
def build_judge_task(candidate_constitution, constitutionName):
    return f"""
Task: Act as an impartial judge.

{build_judge_rubric(constitutionName)}

Evaluate the candidate constitution below against the foundational documents already provided.

Candidate constitution:
{json.dumps(candidate_constitution, indent=2, ensure_ascii=False)}
"""

In [33]:
def anonymize_critique_packet(critique_packet):
    """Model-facing view of the critique packet.

    Replaces the real judge identities with neutral labels ("Judge 1", "Judge 2", ...)
    under a FRESH RANDOM mapping on every call, and shuffles presentation order. This
    prevents a reviser from learning which model (or model family) produced a critique
    (a stable A/O/G-style scheme is learnable across items) and reduces position bias.

    Only used to build the revision prompt sent to a model. The saved packet
    (round_*_critique_packet.json) and everything a human reads keep the real names.
    """
    items = list(critique_packet)
    random.shuffle(items)
    anon = []
    for i, entry in enumerate(items, start=1):
        e = dict(entry)
        e["judge"] = f"Judge {i}"
        anon.append(e)
    return anon


def build_revision_task(current_constitution, critique_packet, round_number, constitutionName):
    return f"""
Task: Act as a constitution writer and reviser.

You are revising the current working constitution for {constitutionName}.

Important instructions:
- Revise the current constitution only where doing so meaningfully improves it.
- Preserve strong existing content.
- Do not rewrite from scratch unless absolutely necessary.
- Do not make the constitution more generic.
- Do not add unnecessary length.
- Apply item-level feedback with explicit operations: add missing coverage; remove invalid
  items; merge redundant items; split compound items; and reword vague items.
- Check the revised set for de facto weighting: several similar criteria must not give one
  principle disproportionate representation merely through repetition.
- When a criterion gives a clear priority or action tendency, preserve serious competing
  considerations that are grounded in the foundational documents. Do not add generic balance.
- Keep the same JSON structure.
- Preserve the four parallel phrasings.
- Make sure each set of four phrasings expresses the same underlying idea.
- Every criterion and guideline keeps its "reasoning" and "questions" fields. If you
  change an item's substance, update its reasoning and questions to match; do not
  regenerate them wholesale for items you did not change.
- Ground all content in the foundational documents already provided.

Current working constitution:
{json.dumps(current_constitution, indent=2, ensure_ascii=False)}

Critique packet from judges:
{json.dumps(anonymize_critique_packet(critique_packet), indent=2, ensure_ascii=False)}

Round number:
{round_number}

Return valid JSON only with exactly this structure:

{CONSTITUTION_JSON_EXAMPLE}
"""

In [34]:
def generate_constitution(model_label, model_id, constitutionName, static_context):
    raw = call_openrouter(
        model=model_id,
        static_context=static_context,
        dynamic_task=build_generation_task(constitutionName),
        model_label=model_label,
        constitutionName=constitutionName,
        temperature=0.7,
    )

    obj = extract_json(raw)
    validate(instance=obj, schema=CONSTITUTION_SCHEMA)
    return obj


def judge_constitution(model_label, model_id, candidate_constitution, constitutionName, static_context):
    raw = call_openrouter(
        model=model_id,
        static_context=static_context,
        dynamic_task=build_judge_task(candidate_constitution, constitutionName),
        model_label=model_label,
        constitutionName=constitutionName,
        temperature=0.3,
    )

    obj = extract_json(raw)
    validate(instance=obj, schema=JUDGE_SCHEMA)
    return obj


def revise_constitution(
    model_label,
    model_id,
    current_constitution,
    critique_packet,
    round_number,
    constitutionName,
    static_context,
    retries=3,
):
    dynamic_task = build_revision_task(
        current_constitution=current_constitution,
        critique_packet=critique_packet,
        round_number=round_number,
        constitutionName=constitutionName,
    )

    last_error = None

    for attempt in range(1, retries + 1):
        raw = None
        try:
            raw = call_openrouter(
                model=model_id,
                static_context=static_context,
                dynamic_task=dynamic_task,
                model_label=model_label,
                constitutionName=constitutionName,
                temperature=0.5,
                max_tokens=8000,
            )

            obj = extract_json(raw)
            validate(instance=obj, schema=CONSTITUTION_SCHEMA)
            return obj

        except Exception as e:
            last_error = e
            print(f"[ERROR] {model_label}Writer revision attempt {attempt}/{retries} failed: {e}")

            save_text(
                f"/content/debug_failed_{constitutionName}_{model_label}_round_{round_number}_attempt_{attempt}.txt",
                raw if "raw" in locals() and raw is not None else "NO RAW MODEL OUTPUT"
            )

            if attempt < retries:
                time.sleep(5)

    raise last_error

In [35]:
def judge_all_models(candidate_constitution, constitutionName, static_context):
    judgments = {}

    for label, model_id in MODELS.items():
        print(f"Judging with {label}Judge...")
        judgments[label] = judge_constitution(
            model_label=label,
            model_id=model_id,
            candidate_constitution=candidate_constitution,
            constitutionName=constitutionName,
            static_context=static_context,
        )

    return judgments


def summarize_judgments(judgments):
    scores = [j["score"] for j in judgments.values()]

    return {
        "average_score": statistics.mean(scores),
        "min_score": min(scores),
        "max_score": max(scores),
        "major_missing_dimension": any(j["major_missing_dimension"] for j in judgments.values()), #if any modl thinks something is missing, marked as true
        "scores_by_judge": {
            label: judgment["score"]
            for label, judgment in judgments.items()
        },
    }


def build_critique_packet(judgments):
    packet = []

    for judge_label, judgment in judgments.items():
        packet.append({
            "judge": judge_label,
            "reasoning": judgment["reasoning"],
            "score": judgment["score"],
            "subscores": judgment["subscores"],
            "major_missing_dimension": judgment["major_missing_dimension"],
            "strengths": judgment["strengths"],
            "weaknesses": judgment["weaknesses"],
            "revision_suggestions": judgment["revision_suggestions"],
        })

    return packet


def passes_consensus(summary):
    return (
        summary["average_score"] >= 93
        and summary["min_score"] >= 90
        and not summary["major_missing_dimension"]
    )

In [36]:
def run_pipeline(constitutionName, static_context, max_rounds=3, output_dir="/content/constitution_results"):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir = Path(output_dir) / f"{constitutionName}_{timestamp}"
    run_dir.mkdir(parents=True, exist_ok=True)

    save_text(run_dir / "static_context.txt", static_context)

    history = {
        "value_system": f"{constitutionName}",
        "run_dir": str(run_dir),
        "initial_candidates": {},
        "initial_candidate_summaries": {},
        "selected_initial_base": None,
        "rounds": [],
    }

    print("\n=== STEP 1: Initial writer generation ===")

    candidates = {}

    for label, model_id in MODELS.items():
        print(f"Generating with {label}Writer...")
        candidates[label] = generate_constitution(label, model_id, constitutionName, static_context)
        save_json(run_dir / f"initial_candidate_{label}.json", candidates[label])

    history["initial_candidates"] = candidates

    print("\n=== STEP 2: Initial judging ===")

    candidate_judgments = {}
    candidate_summaries = {}

    for candidate_label, candidate_constitution in candidates.items():
        print(f"\nJudging initial candidate from {candidate_label}Writer...")

        judgments = judge_all_models(candidate_constitution, constitutionName, static_context)
        summary = summarize_judgments(judgments)

        candidate_judgments[candidate_label] = judgments
        candidate_summaries[candidate_label] = summary

        save_json(run_dir / f"initial_judgments_for_{candidate_label}.json", judgments)
        save_json(run_dir / f"initial_summary_for_{candidate_label}.json", summary)

        print(f"{candidate_label} candidate summary:")
        print(summary)

    base_label = max(
        candidate_summaries,
        key=lambda label: (
            not candidate_summaries[label]["major_missing_dimension"],
            candidate_summaries[label]["average_score"],
            candidate_summaries[label]["min_score"],
        ),
    )

    base_constitution = candidates[base_label]
    base_summary = candidate_summaries[base_label]

    history["initial_candidate_summaries"] = candidate_summaries
    history["selected_initial_base"] = base_label

    print(f"\nSelected base constitution: {base_label}Writer")
    print("Base summary:")
    print(base_summary)

    save_json(run_dir / "selected_initial_base.json", {
        "base_label": base_label,
        "base_summary": base_summary,
        "base_constitution": base_constitution,
    })
    base_judgments = candidate_judgments[base_label]
    base_needs_review = False

    writer_orders = [
        ["Gemini", "GPT", "Claude"],
        ["GPT", "Gemini", "Claude",],
        ["Claude", "Gemini", "GPT"],
    ]

    for round_number in range(1, max_rounds + 1):
        print(f"\n=== ROUND {round_number}: Prepare critique for current base ===")

        if base_needs_review:
            print("Base changed without stored judgments. Rejudging current base...")
            base_judgments = judge_all_models(base_constitution, constitutionName, static_context)
            base_summary = summarize_judgments(base_judgments)
            base_needs_review = False
        else:
            print("Using latest stored judgments for current base.")

        critique_packet = build_critique_packet(base_judgments)

        save_json(run_dir / f"round_{round_number}_base_judgments.json", base_judgments)
        save_json(run_dir / f"round_{round_number}_base_summary.json", base_summary)
        save_json(run_dir / f"round_{round_number}_critique_packet.json", critique_packet)

        print("Current base summary:")
        print(base_summary)

        MIN_REVISIONS = 1
        if round_number > MIN_REVISIONS and passes_consensus(base_summary):
            print("\nConsensus threshold met for current accepted base. Stopping.")
            break
        print(f"\n=== ROUND {round_number}: Sequential writer revision ===")

        revised_constitution = base_constitution
        writer_order = writer_orders[(round_number - 1) % len(writer_orders)]

        for writer_label in writer_order:
            print(f"Revising with {writer_label}Writer...")

            revised_constitution = revise_constitution(
                model_label=writer_label,
                model_id=MODELS[writer_label],
                current_constitution=revised_constitution,
                critique_packet=critique_packet,
                round_number=round_number,
                constitutionName = constitutionName,
                static_context = static_context
            )

            save_json(
                run_dir / f"round_{round_number}_after_{writer_label}Writer.json",
                revised_constitution,
            )

        print(f"\n=== ROUND {round_number}: Judge revised constitution ===")

        revised_judgments = judge_all_models(revised_constitution, constitutionName, static_context)
        revised_summary = summarize_judgments(revised_judgments)

        save_json(run_dir / f"round_{round_number}_revised_judgments.json", revised_judgments)
        save_json(run_dir / f"round_{round_number}_revised_summary.json", revised_summary)

        print("Revised summary:")
        print(revised_summary)

        base_passes = passes_consensus(base_summary)
        revised_passes = passes_consensus(revised_summary)

        improved = (
            (revised_passes and not base_passes)
            or revised_summary["average_score"] > base_summary["average_score"]
            or (
                revised_summary["average_score"] == base_summary["average_score"]
                and revised_summary["min_score"] > base_summary["min_score"]
            )
            or (
                base_summary["major_missing_dimension"]
                and not revised_summary["major_missing_dimension"]
            )
        )

        round_record = {
            "round_number": round_number,
            "writer_order": writer_order,
            "base_summary": base_summary,
            "revised_summary": revised_summary,
            "accepted_revision": improved,
        }

        history["rounds"].append(round_record)
        save_json(run_dir / f"round_{round_number}_record.json", round_record)

        if improved:
            print("Accepted revised constitution as new base.")
            base_constitution = revised_constitution
            base_summary = revised_summary
            base_judgments = revised_judgments
            base_needs_review = False
        else:
            print("Rejected revised constitution. Keeping previous base.")
            base_needs_review = False

        if round_number >= MIN_REVISIONS and passes_consensus(base_summary):
            print("\nConsensus threshold met after revision. Stopping.")
            break

    print("\n=== FINAL RESULT ===")

    final_judgments = base_judgments
    final_summary = base_summary

    history["final_constitution"] = base_constitution
    history["final_judgments"] = final_judgments
    history["final_summary"] = final_summary

    save_json(run_dir / "final_constitution.json", base_constitution)
    save_json(run_dir / "final_judgments.json", final_judgments)
    save_json(run_dir / "final_summary.json", final_summary)
    save_json(run_dir / "run_history.json", history)

    print("\nFinal summary:")
    print(final_summary)
    print(f"\nSaved results to: {run_dir}")

    return history

In [37]:
def run_everything(constitutionName, docs_root, max_rounds):
    docs_folder = Path(docs_root) / f"{constitutionName} Docs"
    foundationalDocs = load_markdown_docs(str(docs_folder))
    static_context = build_static_context(constitutionName, foundationalDocs)

    return run_pipeline(
        constitutionName=constitutionName,
        static_context=static_context,
        max_rounds=max_rounds,
        output_dir=f"/content/{constitutionName}_constitution_results",
    )